# Part 5 - GRU Sequence Models on CIFAR-10
This notebook compares two sequence representations (row-wise and patch-wise) and plots combined curves like Khoa's section.

In [ ]:
import time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3
NUM_CLASSES = 10
NUM_WORKERS = 2
PATCH = 4


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_ds = torchvision.datasets.CIFAR10(
    "./data", train=True, download=True, transform=transform
)
test_ds = torchvision.datasets.CIFAR10(
    "./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

print("Train size:", len(train_ds))
print("Test size:", len(test_ds))
print("Classes:", train_ds.classes)


In [ ]:
def to_row_sequence(x):
    B, C, H, W = x.shape
    return x.permute(0, 2, 1, 3).contiguous().view(B, H, C * W)

def to_patch_sequence(x, p=4):
    B, C, H, W = x.shape
    assert H % p == 0 and W % p == 0
    h, w = H // p, W // p
    x = x.view(B, C, h, p, w, p).permute(0, 2, 4, 1, 3, 5).contiguous()
    tokens = x.view(B, h * w, C * p * p)
    return tokens


In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=10):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.gru(x)
        last = out[:, -1, :]
        return self.fc(last)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, mode="row"):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        if mode == "row":
            seq = to_row_sequence(imgs)
        else:
            seq = to_patch_sequence(imgs, p=PATCH)

        optimizer.zero_grad()
        out = model(seq)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        pred = out.argmax(1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, mode="row"):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        if mode == "row":
            seq = to_row_sequence(imgs)
        else:
            seq = to_patch_sequence(imgs, p=PATCH)

        out = model(seq)
        loss = criterion(out, labels)

        total_loss += loss.item() * imgs.size(0)
        pred = out.argmax(1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


In [ ]:
def run(mode="row"):
    if mode == "row":
        input_size = 3 * 32
    else:
        input_size = 3 * (PATCH * PATCH)

    model = GRUClassifier(input_size=input_size, hidden_size=128).to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    start = time.time()

    for e in range(EPOCHS):
        tl, ta = train_one_epoch(model, train_loader, crit, opt, device, mode)
        vl, va = evaluate(model, test_loader, crit, device, mode)

        history["train_loss"].append(tl)
        history["train_acc"].append(ta)
        history["test_loss"].append(vl)
        history["test_acc"].append(va)

        print(
            f"{mode} | Epoch {e+1}/{EPOCHS} | "
            f"Train Loss: {tl:.4f} | Train Acc: {ta:.4f} | "
            f"Test Loss: {vl:.4f} | Test Acc: {va:.4f}"
        )

    total_time = time.time() - start

    result = {
        "final_train_loss": history["train_loss"][-1],
        "final_train_acc": history["train_acc"][-1],
        "final_test_loss": history["test_loss"][-1],
        "final_test_acc": history["test_acc"][-1],
        "train_time_sec": total_time,
        "epochs": EPOCHS,
        "mode": mode
    }
    return result, history


In [ ]:
print("Training GRU (row-wise)")
res_row, hist_row = run("row")

print("\nTraining GRU (patch-wise)")
res_patch, hist_patch = run("patch")


In [ ]:
def plot_comparison(hist1, hist2, label1="GRU Row-wise", label2="GRU Patch-wise"):
    epochs = range(1, len(hist1["train_loss"]) + 1)

    plt.figure(figsize=(6, 4))
    plt.plot(epochs, hist1["train_loss"], label=label1)
    plt.plot(epochs, hist2["train_loss"], label=label2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss by Epoch")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(epochs, hist1["test_loss"], label=label1)
    plt.plot(epochs, hist2["test_loss"], label=label2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Test Loss by Epoch")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(epochs, hist1["train_acc"], label=label1)
    plt.plot(epochs, hist2["train_acc"], label=label2)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training Accuracy by Epoch")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(epochs, hist1["test_acc"], label=label1)
    plt.plot(epochs, hist2["test_acc"], label=label2)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Test Accuracy by Epoch")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
plot_comparison(hist_row, hist_patch, "GRU Row-wise", "GRU Patch-wise")


In [ ]:
print("\nTable-ready results (same format as Khoa)")
print("-" * 120)
print(f"{'Model':30s} {'Train Accuracy':15s} {'Test Accuracy':15s} {'Train Loss':12s} {'Test Loss':12s} {'Training Time (s)':18s}")
print("-" * 120)

print(
    f"{'GRU (row-wise)':30s} "
    f"{res_row['final_train_acc']:.4f}           "
    f"{res_row['final_test_acc']:.4f}           "
    f"{res_row['final_train_loss']:.4f}       "
    f"{res_row['final_test_loss']:.4f}       "
    f"{res_row['train_time_sec']:.2f}"
)

print(
    f"{'GRU (patch-wise)':30s} "
    f"{res_patch['final_train_acc']:.4f}           "
    f"{res_patch['final_test_acc']:.4f}           "
    f"{res_patch['final_train_loss']:.4f}       "
    f"{res_patch['final_test_loss']:.4f}       "
    f"{res_patch['train_time_sec']:.2f}"
)
